# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [3]:
!python3 -m pip install -qU jq lark langchain langchain-elasticsearch langchain_openai tiktoken

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

Python est introuvable ; ex�cutez sans arguments � installer � partir du Microsoft Store ou d�sactivez ce raccourci dans Param�tres > Applications > param�tres d


## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial. 

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [5]:
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name="ironhack", #give it a meaningful name,
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [6]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [9]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    # Populate the metadata dictionary with keys name, summary, url, category, and updated_at
    metadata["name"] = record.get("name")
    metadata["summary"] = record.get("summary")
    metadata["url"] = record.get("url")
    metadata["category"] = record.get("category")
    metadata["updated_at"] = record.get("updated_at")

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400 #define chunk size and chunk overlap
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [10]:
documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name="ironhack",  # use the same index name as above
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [11]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the sales team at NASA?', '2. How does the sales team operate within NASA?', '3. What are the responsibilities of the NASA sales team?']


---- Answer ----
The NASA sales team is a part of the Americas region in the sales organization of the company. It is led by two Area Vice-Presidents, Laura Martinez for North America and Gary Johnson for South America. The team is responsible for promoting and selling the company's products and services in the North and South American markets. They work closely with other departments, such as marketing, product development, and customer support, to ensure the company's success in these regions.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

In [13]:
ans = chain.invoke("what is the purpose of onfidentiality in Data Security?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What is the significance of confidentiality in Data Security?', '2. How does confidentiality play a role in Data Security?', '3. Why is confidentiality important for Data Security?']


---- Answer ----

The purpose of confidentiality in data security is to protect sensitive information from unauthorized access, use, or disclosure. This includes safeguarding the organization's confidential information, as well as any sensitive information entrusted to employees by clients, partners, or other third parties. Confidentiality helps maintain the trust of stakeholders and prevents potential harm to the organization, its employees, and its clients. It also ensures compliance with data protection policies and regulations.


In [14]:
ans = chain.invoke("what can you say about the programming language Solidity?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What information can you provide about the programming language Solidity?', '2. Can you give me any insights on the programming language Solidity?', '3. How would you describe the programming language Solidity?']


---- Answer ----
Solidity is a high-level programming language used for writing smart contracts on the Ethereum blockchain. It is an object-oriented language that is influenced by C++, Python, and JavaScript. Solidity is designed to be secure, as it is used to write code that controls digital assets on the blockchain. It is also Turing complete, meaning it can handle any computation that can be performed by a computer. Solidity is a popular choice for developers working on decentralized applications (dApps) and has a growing community of developers and resources available for learning and support.


In [16]:
ans = chain.invoke("what can you say about the programming language COBOL? What about Solidity?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the programming language COBOL and its features?', '2. How does the programming language COBOL compare to other languages in terms of functionality and usage?', '3. What are the key differences between the programming language COBOL and Solidity?']


---- Answer ----

COBOL (Common Business Oriented Language) is a high-level programming language that was developed in the late 1950s. It was primarily used for business and administrative applications, and was one of the first programming languages to be widely used in the industry. Despite its age, COBOL is still used in many legacy systems and is known for its readability and ability to handle large amounts of data.

Solidity, on the other hand, is a relatively new programming language that was created in 2014. It is primarily used for developing smart contracts on the Ethereum blockchain. Solidity is a high-level, object-oriented language that is similar to JavaScript in syntax. It is known for its security features and is becoming increasingly popular as the use of blockchain technology grows.


### My Observations:
This lab really opened my eyes to how much more sophisticated retrieval can be beyond simple vector similarity search. Watching the MultiQuery Retriever automatically generate alternative phrasings of my questions was fascinating - like when I asked about "Solidity" and it created variations like "insights on Solidity" and "describe Solidity." The logging output showing these generated queries made it clear how the system is casting a wider net to capture more relevant documents. I was impressed by how well it handled both technical queries about programming languages and business-oriented questions about team structures.

### Key Takeaway:
The biggest insight was understanding that users often don't phrase questions optimally for retrieval systems. The MultiQuery approach solves this by having the LLM itself generate better variations of the original question, essentially using AI to improve AI retrieval. My comparative question about "COBOL vs Solidity" particularly demonstrated this - the system generated queries that looked at both languages individually and comparatively. This pattern of using LLMs to enhance retrieval quality seems like a game-changer for building more robust RAG systems. It's a clever solution to the problem of query formulation, which I hadn't even realized was a limitation until seeing this approach in action.